# AInstein component: LLM QA

In [1]:
import sys
from pathlib import Path

path_project = Path.cwd().parent
sys.path.append(str(path_project))

In [2]:
from AInstein import (
    get_llm_azure_openai, # conexión al modelo GPT
    get_settings, # conexión a las configuraciones de los modelos
)

In [3]:
import re
import json
from collections import defaultdict

import ipywidgets as widgets
import pandas as pd
from IPython.display import display, Markdown, HTML
from openpyxl import load_workbook

# Settings

In [5]:
# Settings
ENVIRONMENT: str = 'bdb-gcp-sbx-ia'
WORKPLACE_PROJECT_ID: str = 'geo-cargas_laborales'

# Obtener las configuraciones del proyecto
settings = get_settings(WORKPLACE_PROJECT_ID, environment=ENVIRONMENT)

# Models

In [6]:
# Crear instancias de los modelos
llm = get_llm_azure_openai(settings)  # Modelo de Azure OpenAI

# Responses

## General

In [108]:
class ProcesadorTranscripcionTeams:
    """
    Procesa archivos VTT de Microsoft Teams para levantar actividades,
    enriquecerlas con IA y generar resúmenes confirmables.
    """

    # ------------------------------------------------------------------
    # 1. LECTURA Y LIMPIEZA DEL VTT
    # ------------------------------------------------------------------
    def leer_archivo_vtt(self, contenido):
        conversaciones = []
        lineas = contenido.splitlines()

        timestamp_actual = None
        buffer_texto = []

        for linea in lineas:
            linea = linea.strip()

            if (
                not linea
                or linea == "WEBVTT"
                or re.match(r"^[a-f0-9\-]+\/\d+\-\d+$", linea)
            ):
                continue

            if "-->" in linea:
                if timestamp_actual and buffer_texto:
                    conversaciones.append({
                        "timestamp": timestamp_actual,
                        "texto": " ".join(buffer_texto).strip()
                    })
                    buffer_texto = []

                inicio = linea.split("-->")[0].strip()
                timestamp_actual = inicio.split(".")[0]
                continue

            linea = re.sub(r"<v[^>]*>", "", linea)
            linea = re.sub(r"</v>", "", linea)
            linea = re.sub(r"^[A-Za-zÁÉÍÓÚÑáéíóúñ\s,]+:\s*", "", linea)

            if linea:
                buffer_texto.append(linea)

        if timestamp_actual and buffer_texto:
            conversaciones.append({
                "timestamp": timestamp_actual,
                "texto": " ".join(buffer_texto).strip()
            })

        return conversaciones

    # ------------------------------------------------------------------
    # 2. EXTRACCIÓN DE ACTIVIDADES (SIN IA)
    # ------------------------------------------------------------------
    def extraer_actividades(self, conversaciones):
        actividades = []
        actividad_actual = None

        patrones_inicio = [
            r"\binicio\b",
            r"\biniciar\b",
            r"\bcomienzo\b",
            r"\bempiezo\b",
            r"\bvoy a iniciar\b"
        ]

        patrones_fin = [
            r"\bfinalizo\b",
            r"\btermino\b",
            r"\bterminar\b",
            r"\bfinalizar\b"
        ]

        for conv in conversaciones:
            texto = conv["texto"].lower()
            timestamp = conv["timestamp"]

            if any(re.search(p, texto) for p in patrones_inicio):
                actividad_actual = {
                    "descripcion": conv["texto"],
                    "inicio": timestamp,
                    "fin": None
                }

            elif actividad_actual and any(re.search(p, texto) for p in patrones_fin):
                actividad_actual["fin"] = timestamp
                actividades.append(actividad_actual)
                actividad_actual = None

        return actividades
    
    # ------------------------------------------------------------------
    # 3. CÁLCULO DE DURACIÓN
    # ------------------------------------------------------------------
    def calcular_duracion_minutos(self, inicio, fin):
        h1, m1, s1 = map(int, inicio.split(":"))
        h2, m2, s2 = map(int, fin.split(":"))

        t1 = h1 * 3600 + m1 * 60 + s1
        t2 = h2 * 3600 + m2 * 60 + s2

        segundos = t2 - t1
        return max(segundos // 60, 0)

    # ------------------------------------------------------------------
    # 4. PIPELINE DE PROCESAMIENTO
    # ------------------------------------------------------------------
    def procesar_archivo(self, contenido):
        conversaciones = self.leer_archivo_vtt(contenido)
        actividades = self.extraer_actividades(conversaciones)

        resultado = []
        for a in actividades:
            if not a["fin"]:
                continue

            duracion = self.calcular_duracion_minutos(a["inicio"], a["fin"])
            if duracion <= 0:
                continue

            resultado.append({
                "actividad": a["descripcion"],
                "inicio": a["inicio"],
                "fin": a["fin"],
                "duracion_min": duracion
            })

        return resultado

    # ------------------------------------------------------------------
    # 5. ENRIQUECIMIENTO CON IA (UNIDAD + PHVA)
    # ------------------------------------------------------------------
    def _parse_json_seguro(self, texto):
        texto = texto.strip()
    
        if not texto:
            raise ValueError("❌ El LLM devolvió una respuesta vacía")
    
        # Quitar markdown si aparece
        if texto.startswith("```"):
            texto = re.sub(r"```json|```", "", texto).strip()
    
        # Recortar hasta el primer [ o {
        inicio = min(
            [i for i in [texto.find("["), texto.find("{")] if i != -1],
            default=-1
        )
    
        if inicio > 0:
            texto = texto[inicio:]
    
        try:
            return json.loads(texto)
        except json.JSONDecodeError:
            print("❌ JSON inválido devuelto por el LLM")
            print("Respuesta cruda:")
            print(texto)
            raise

    def enriquecer_actividades(self, actividades, cargo, vicepresidencia):
    
        prompt = f"""
    Eres un analista experto en Levantamiento de Cargas Laborales.
    
    Contexto del colaborador:
    - Cargo: {cargo}
    - Vicepresidencia: {vicepresidencia}
    
    Para cada actividad, debes:
    - unidad_medida (ej: solicitudes, informes, reuniones, casos, desarrollos)
    - phva (Planear, Hacer, Verificar, Actuar)
    
    ⚠️ REGLAS ESTRICTAS:
    - Devuelve EXCLUSIVAMENTE un JSON válido
    - NO incluyas texto antes o después
    - NO expliques nada
    - NO uses markdown
    - Devuelve una LISTA del mismo tamaño que la entrada
    
    Formato exacto de salida:
    [
      {{
        "nombre": "texto corto",
        "unidad_medida": "texto",
        "phva": "Planear | Hacer | Verificar | Actuar"
      }}
    ]
    
    Actividades de entrada:
    {json.dumps(actividades, indent=2, ensure_ascii=False)}
    """
        reply = llm.invoke(prompt)
        return self._parse_json_seguro(reply.content)

    # ------------------------------------------------------------------
    # 6. CONSTRUCCIÓN DE RESUMEN ESTRUCTURADO
    # ------------------------------------------------------------------
    def construir_resumen_actividades(self, actividades_enriquecidas):
        resumen = ""

        for i, act in enumerate(actividades_enriquecidas, start=1):
            resumen += f"""
Actividad {i}:
- Descripción: {act['nombre']}
- Frecuencia: {act.get('frecuencia', 'No especificada')}
- Duración (minutos): {act.get('duracion_min')}
- Proceso del área: {act.get('proceso_area')}
- Volumen: {act.get('volumen', 'N/A')}
- Unidad de Medida: {act.get('unidad_medida')}
- Tipo de actividad (PHVA): {act.get('phva')}
- Autonomía: {act.get('autonomia', 'N/A')}%
"""
            if act.get("observaciones"):
                resumen += f"- Observaciones: {act['observaciones']}\n"

        return resumen

    # ------------------------------------------------------------------
    # 7. RESUMEN PARA CONFIRMACIÓN DEL USUARIO
    # ------------------------------------------------------------------
    def resumen_para_confirmacion(
        self,
        #actividades,
        actividades_enriquecidas,
        #inputs_usuario,
        contexto
    ):
        resumen_actividades = self.construir_resumen_actividades(
            actividades_enriquecidas
        )

        prompt = f"""
Eres un asistente de Levantamiento de Cargas Laborales.

Contexto del colaborador:
- Cargo: {contexto['cargo']}
- Vicepresidencia: {contexto['vicepresidencia']}

A continuación se presenta un resumen de las actividades
identificadas a partir de reuniones y la información ingresada
por el colaborador.

{resumen_actividades}

Instrucciones:
- Resume la información de forma clara y ordenada
- No combines el Volumen con la Unidad de Medida, 
dalos por separado ya que el volumen es respecto a la Frecuencia (ej: Volumen = 1 y Frecuencia = Diaria -> 1 vez al día)
- Usa lenguaje sencillo y no técnico
- No agregues información nueva
- No hagas cálculos adicionales
- Finaliza preguntando si la información es correcta o si desea hacer ajustes
"""
        reply = llm.invoke(prompt)
        return reply.content

    # ------------------------------------------------------------------
    # 8. LECTURA DE ARCHIVO DESDE FILEUPLOAD (JUPYTER)
    # ------------------------------------------------------------------
    def obtener_contenido_vtt(self, upload_widget):
        if not upload_widget.value:
            raise ValueError("No se ha subido ningún archivo")

        valor = upload_widget.value
        archivo = valor[0] if isinstance(valor, tuple) else list(valor.values())[0]
        contenido_raw = archivo["content"]

        if isinstance(contenido_raw, memoryview):
            return contenido_raw.tobytes().decode("utf-8")
        elif isinstance(contenido_raw, bytes):
            return contenido_raw.decode("utf-8")
        else:
            raise TypeError("Tipo de contenido no soportado")

     # ------------------------------------------------------------------
     # 8. LECTURA DE ARCHIVO DESDE FILEUPLOAD (JUPYTER)
     # ------------------------------------------------------------------
    def fusionar_inputs_usuario(self, actividades_base, actividades_enriquecidas, inputs_usuario):
        """
        Une actividades detectadas + enriquecidas por IA + inputs del usuario
        """
        actividades_finales = []
        
        for base, ia, user in zip(
            actividades_base,
            actividades_enriquecidas,
            inputs_usuario
        ):
            actividades_finales.append({
                "nombre": ia["nombre"],
                "inicio": base["inicio"],
                "fin": base["fin"],
                "duracion_min": base["duracion_min"],
        
                "unidad_medida": ia["unidad_medida"],
                "phva": ia["phva"],
        
                "frecuencia": user["frecuencia"],
                "volumen": user["volumen"],
                "proceso_area": user["proceso_area"],
                "autonomia": user["autonomia"],
                "observaciones": user.get("observaciones", "")
            })
        
        return actividades_finales

    # ------------------------------------------------------------------
     # 9. CONFIRMAR LA INFORMACIÓN POR PARTE DEL USUARIO
     # ------------------------------------------------------------------
    def confirmar_informacion(self, resumen_texto):
        print("📋 RESUMEN PARA CONFIRMACIÓN\n")
        print(resumen_texto)
        
        respuesta = input("\n¿La información es correcta? (si / no): ").strip().lower()
        return respuesta == "si"

    # ------------------------------------------------------------------
     # 10. CALCULAR LAS MÉTRICAS (falta poner el resto de cálculos)
     # -----------------------------------------------------------------
    def calcular_metricas(self, actividades):
        """
        Calcula métricas globales de carga laboral integrando:
        - tiempo
        - volumen
        - frecuencia
        - autonomía
        - PHVA
        - proceso / área
    
        Modelo alineado con estudio de cargas laborales.
        """
    
        # Constantes
        DIAS_LABORALES_MES = 21
        MINUTOS_JORNADA_MES = 220 * 60
        JORNADA_LABORAL_DIARIA = 8.5
    
        FACTOR_FRECUENCIA = {
            "Diario": 1,
            "Semanal": 1 / 5,
            "Quincenal": 1 / 10,
            "Mensual": 1 / 21,
            "Bimensual": 1 / 42,
            "Trimestral": 1 / 63,
            "Semestral": 1 / 126,
            "Anual": 1 / 252
        }
    
        # Acumuladores
        carga_w_por_phva = {}
        cantidad_por_phva = {}
    
        carga_w_por_frecuencia = {}
        cantidad_por_frecuencia = {}
    
        carga_w_por_proceso_area = {}
        cantidad_por_proceso_area = {}
    
        total_carga_w_sin_tm = 0
        total_carga_trabajo_individual = 0
        total_minutos_diarios = 0
    
        # Procesamiento Actividad
        for act in actividades:
            tiempo_base = act["duracion_min"]
            volumen = act["volumen"]
            frecuencia = act["frecuencia"]
            autonomia = act["autonomia"] / 100
    
            phva = act.get("phva", "SIN CLASIFICAR")
            proceso_area = act.get("proceso_area", "SIN CLASIFICAR")
    
            factor = FACTOR_FRECUENCIA.get(frecuencia, 0)
    
            minutos_diarios = tiempo_base * volumen * factor
            minutos_mes = minutos_diarios * DIAS_LABORALES_MES
    
            carga_w_sin_tm = minutos_mes / MINUTOS_JORNADA_MES
            carga_trabajo_individual = carga_w_sin_tm * autonomia
    
            # Guardar métricas por actividad (opcional pero MUY útil)
            act["metricas"] = {
                "minutos_diarios": round(minutos_diarios, 2),
                "minutos_mes": round(minutos_mes, 2),
                "carga_w_sin_tm": round(carga_w_sin_tm, 4),
                "carga_trabajo_individual": round(carga_trabajo_individual, 4)
            }
    
            total_carga_w_sin_tm += carga_w_sin_tm
            total_carga_trabajo_individual += carga_trabajo_individual
            total_minutos_diarios += minutos_diarios
    
            # ---- PHVA ----
            carga_w_por_phva[phva] = carga_w_por_phva.get(phva, 0) + carga_w_sin_tm
            cantidad_por_phva[phva] = cantidad_por_phva.get(phva, 0) + 1
    
            # ---- Frecuencia ----
            carga_w_por_frecuencia[frecuencia] = (
                carga_w_por_frecuencia.get(frecuencia, 0) + carga_w_sin_tm
            )
            cantidad_por_frecuencia[frecuencia] = (
                cantidad_por_frecuencia.get(frecuencia, 0) + 1
            )
    
            # ---- Proceso / Área ----
            carga_w_por_proceso_area[proceso_area] = (
                carga_w_por_proceso_area.get(proceso_area, 0) + carga_w_sin_tm
            )
            cantidad_por_proceso_area[proceso_area] = (
                cantidad_por_proceso_area.get(proceso_area, 0) + 1
            )
    
        # Tiempo Muerto
        almuerzo = 1 / 8
        baño_agua_etc = (3 * 5) / 60
        break_15min = 15 / 60
        latencia_software = 10 / 60
        cliente_ext_o_int = 20 / 60
    
        tiempo_muerto = sum([
            almuerzo / JORNADA_LABORAL_DIARIA,
            baño_agua_etc / JORNADA_LABORAL_DIARIA,
            break_15min / JORNADA_LABORAL_DIARIA,
            latencia_software / JORNADA_LABORAL_DIARIA,
            cliente_ext_o_int / JORNADA_LABORAL_DIARIA
        ])
    
        factor_tiempo_neto = round(1 - tiempo_muerto, 4)
    
        # Porcentajes
        def porcentajes(dic):
            total = sum(dic.values())
            return {
                k: round(v / total, 4) if total > 0 else 0
                for k, v in dic.items()
            }
    
        # Resultado Final    
        horas_diarias_requeridas = total_minutos_diarios / 60
        horas_netas_por_persona = JORNADA_LABORAL_DIARIA * factor_tiempo_neto
    
        return {
            # ---- PHVA ----
            "carga_trabajo_phva": {k: round(v, 4) for k, v in carga_w_por_phva.items()},
            "cantidad_actividades_phva": cantidad_por_phva,
            "porcentaje_actividades_phva": porcentajes(carga_w_por_phva),
    
            # ---- Frecuencia ----
            "carga_trabajo_frecuencia": {k: round(v, 4) for k, v in carga_w_por_frecuencia.items()},
            "cantidad_actividades_frecuencia": cantidad_por_frecuencia,
            "porcentaje_actividades_frecuencia": porcentajes(carga_w_por_frecuencia),
    
            # ---- Proceso / Área ----
            "carga_trabajo_proceso_area": {
                k: round(v, 4) for k, v in carga_w_por_proceso_area.items()
            },
            "cantidad_actividades_proceso_area": cantidad_por_proceso_area,
            "porcentaje_actividades_proceso_area": porcentajes(carga_w_por_proceso_area),
    
            # ---- Totales ----
            "total_carga_w_sin_tm": round(total_carga_w_sin_tm, 4),
            "total_carga_trabajo_individual": round(total_carga_trabajo_individual, 4),
            "minutos_diarios_empleados": round(total_minutos_diarios, 2),
            "horas_diarias_requeridas": round(horas_diarias_requeridas, 2),
    
            # ---- Dotación ----
            "jornada_laboral_diaria": JORNADA_LABORAL_DIARIA,
            "factor_tiempo_neto_productivo": factor_tiempo_neto,
            "horas_netas_efectivas_por_persona": round(horas_netas_por_persona, 2),
            "numero_personas_requeridas": round(
                horas_diarias_requeridas / horas_netas_por_persona, 3
            ),
    
            # ---- Ajuste final ----
            "tiempo_muerto": round(tiempo_muerto, 4),
            "horas_diarias_requeridas_final": round(
                total_minutos_diarios * (1 + tiempo_muerto) / 60, 2
            )
        }
    
    # ------------------------------------------------------------------
     # 11. ANÁLISIS DE LA IA PARA EL ANALISTA 
     # ------------------------------------------------------------------
    def analisis_analista_ia(self, actividades, metricas, contexto):
        prompt = f"""
    Eres un ANALISTA SENIOR en Levantamiento de Cargas Laborales y Dimensionamiento Operativo.
    Tu tarea NO es recalcular datos, sino INTERPRETAR, VALIDAR COHERENCIA y EMITIR JUICIO PROFESIONAL
    a partir de la información suministrada.
    
    =========================
    CONTEXTO ORGANIZACIONAL
    =========================
    Cargo: {contexto.get("cargo")}
    Vicepresidencia: {contexto.get("vicepresidencia")}
    
    =========================
    ACTIVIDADES ANALIZADAS
    =========================
    Cada actividad incluye duración, frecuencia, volumen, autonomía, PHVA y proceso/área.
    
    {json.dumps(actividades, indent=2, ensure_ascii=False)}
    
    =========================
    MÉTRICAS CALCULADAS
    =========================
    Las métricas ya incluyen:
    - Carga diaria y mensual
    - Distribución PHVA
    - Distribución por frecuencia
    - Autonomía promedio
    - Cálculo de dotación considerando tiempos muertos
    
    {json.dumps(metricas, indent=2, ensure_ascii=False)}
    
    =========================
    INSTRUCCIONES DE ANÁLISIS
    =========================
    
    1. Analiza el NIVEL DE CARGA LABORAL:
       - Usa las horas diarias requeridas finales y la dotación calculada.
       - Clasifica la carga como BAJA, ADECUADA o ALTA.
       - Justifica la clasificación con base en capacidad operativa real.
    
    2. Evalúa la COHERENCIA del resultado:
       - ¿Las métricas son consistentes con el tipo de cargo?
       - ¿Existen valores atípicos o concentraciones anómalas de carga?
       - ¿La autonomía declarada es coherente con la carga individual?
    
    3. Interpreta el BALANCE PHVA:
       - Explica qué indica la distribución PHVA sobre la naturaleza del rol.
       - Identifica posibles desviaciones (sobregestión, baja ejecución, ausencia de mejora).
    
    4. Identifica RIESGOS OPERATIVOS:
       - Riesgos de sobrecarga, dependencia, reprocesos o cuellos de botella.
       - Riesgos derivados de frecuencia, volumen o baja autonomía.
    
    5. Detecta OPORTUNIDADES DE AUTOMATIZACIÓN O MEJORA:
       - Actividades repetitivas, de alta frecuencia o bajo valor agregado.
       - Actividades con alta carga y baja autonomía.
    
    6. Formula RECOMENDACIONES EJECUTIVAS:
       - Orientadas a gestión del cargo, redistribución de carga o mejora operativa.
       - No propongas soluciones técnicas específicas, solo lineamientos.
    
    =========================
    FORMATO DE RESPUESTA (OBLIGATORIO)
    =========================
    
    Responde en español, con lenguaje profesional, estructurado en los siguientes bloques:
    
    1. Nivel de carga laboral (con justificación)
    2. Validación de coherencia de las métricas
    3. Interpretación del balance PHVA
    4. Riesgos operativos identificados
    5. Oportunidades de automatización o mejora
    6. Recomendaciones finales
    
    Quiero que menciones los valores numéricos mientras vas dando el análisis
    Interpreta y explica su significado.
    
    """
    
        reply = llm.invoke(prompt)
        
        return reply.content

    # ------------------------------------------------------------------
     # 12. CAPTURAR EL CARGO Y LA VICEPRESIDENCIA POR PARTE DEL USUARIO
     # ------------------------------------------------------------------
    def capturar_contexto_usuario(self):
        cargo = widgets.Text(
            description="Cargo:",
            placeholder="Ej: Analista de Datos",
            style={'description_width': '120px'},
            layout=widgets.Layout(width='400px')
        )
    
        vicepresidencia = widgets.Text(
            description="Vicepresidencia:",
            placeholder="Ej: Tecnología",
            style={'description_width': '120px'},
            layout=widgets.Layout(width='400px')
        )
    
        display(cargo, vicepresidencia)
    
        return {
            "cargo": cargo,
            "vicepresidencia": vicepresidencia
        }

    # ------------------------------------------------------------------
     # 13. OBTENER EL CARGO Y LA VICEPRESIDENCIA
     # ------------------------------------------------------------------
    def obtener_contexto_valores(self, contexto_widgets):
        return {
            "cargo": contexto_widgets["cargo"].value,
            "vicepresidencia": contexto_widgets["vicepresidencia"].value
        }

    # ----------------------------------------------------------------------
     # 14. CAPTURAR EL RESTO DE INPUTS POR ACTIVIDAD POR PARTE DEL USUARIO
     # ---------------------------------------------------------------------
    def capturar_inputs_usuario(self):
        frecuencia = widgets.Dropdown(
            options=["Diario", "Mensual", "Semanal", "Quincenal", "Bimensual", "Trimestral", "Semestral", "Anual"],
            description="Frecuencia:",
            style={'description_width': '120px'}
        )
    
        volumen = widgets.IntText(
            description="Volumen:",
            value=1,
            style={'description_width': '120px'}
        )
    
        autonomia = widgets.IntSlider(
            description="Autonomía (%):",
            min=0,
            max=100,
            value=80,
            style={'description_width': '120px'}
        )

        proceso_area = widgets.Textarea(
            description="Proceso del área:",
            layout=widgets.Layout(width='500px', height='80px'),
            style={'description_width': '120px'}
        )
    
        observaciones = widgets.Textarea(
            description="Observaciones:",
            layout=widgets.Layout(width='500px', height='80px'),
            style={'description_width': '120px'}
        )
    
        display(frecuencia, volumen, autonomia, proceso_area, observaciones)
    
        return {
            "frecuencia": frecuencia,
            "volumen": volumen,
            "proceso_area": proceso_area,
            "autonomia": autonomia,
            "observaciones": observaciones
        }

    def obtener_actividades_unicas(self, actividades):
        vistas = set()
        unicas = []

        for act in actividades:
            nombre = act["actividad"].strip().lower()
            if nombre not in vistas:
                vistas.add(nombre)
                unicas.append(act)
        return unicas

    def capturar_inputs_usuario_actividades_unicas(self, actividades):
        actividades_unicas = self.obtener_actividades_unicas(actividades)

        inputs_por_actividad = {}

        for act in actividades_unicas:
            print(f"\n Actividad: {act['actividad']}")
            inputs_por_actividad[act["actividad"].strip().lower()] = (
                self.capturar_inputs_usuario()
            )

        return inputs_por_actividad

    # ------------------------------------------------------------------
     # 15. OBTENER EL RESTO DE INPUTS
     # ------------------------------------------------------------------
    def obtener_inputs_usuario_valores(self, inputs_widgets):
        return {
            "frecuencia": inputs_widgets["frecuencia"].value,
            "volumen": inputs_widgets["volumen"].value,
            "autonomia": inputs_widgets["autonomia"].value,
            "proceso_area": inputs_widgets["proceso_area"].value,
            "observaciones": inputs_widgets["observaciones"].value
        }

    # ------------------------------------------------------------------
     # 16. LECTURA DE ARCHIVO DESDE FILEUPLOAD (JUPYTER)
     # ------------------------------------------------------------------
    def editar_actividades(self, actividades):
        """
        Permite al usuario modificar campos de una actividad específica.
        """
        while True:
            print("\n✏️ ACTIVIDADES DISPONIBLES:")
            for i, act in enumerate(actividades, start=1):
                print(f"{i}. {act['nombre']}")
    
            opcion = input(
                "\nIngrese el número de la actividad a modificar "
                "(o 'salir' para terminar ajustes): "
            ).strip().lower()
    
            if opcion == "salir":
                break
    
            if not opcion.isdigit() or not (1 <= int(opcion) <= len(actividades)):
                print("⚠️ Opción inválida")
                continue
    
            idx = int(opcion) - 1
            actividad = actividades[idx]
    
            print("\nCampos editables:")
            campos_editables = [
                "frecuencia",
                "volumen",
                "duracion_min",
                "autonomia",
                "observaciones",
                "unidad_medida",
                "phva"
            ]
    
            for c in campos_editables:
                print(f"- {c} (actual: {actividad.get(c)})")
    
            campo = input("\nCampo a modificar: ").strip()
    
            if campo not in campos_editables:
                print("⚠️ Campo no editable")
                continue
    
            nuevo_valor = input("Nuevo valor: ").strip()
    
            # Cast simple
            if campo in ("volumen", "autonomia_pct"):
                nuevo_valor = int(nuevo_valor)
    
            actividad[campo] = nuevo_valor
            print("✅ Campo actualizado correctamente")


    # ------------------------------------------------------------------
    # 17. CAPTURA DE ACTIVIDADES NO DIARIAS (MANUAL)
    # ------------------------------------------------------------------
    def capturar_actividades_no_diarias(self):
        """
        Permite al usuario agregar actividades que NO ocurren diariamente
        (semanales, mensuales, trimestrales, etc.).
        """
    
        actividades = []
    
        print("\n📌 REGISTRO DE ACTIVIDADES NO DIARIAS")
        print("Escriba 'salir' cuando no desee agregar más actividades.\n")
    
        while True:
            nombre = input("Nombre de la actividad: ").strip()
            if nombre.lower() == "salir":
                break
    
            frecuencia = input(
                "Frecuencia (Semanal / Mensual / Trimestral / etc.): "
            ).strip()
    
            duracion_min = input(
                "Duración estimada por ejecución (en minutos): "
            ).strip()
    
            volumen = input(
                "Volumen por periodo (ej: 1, 2, 5): "
            ).strip()
    
            autonomia = input(
                "Autonomía (%): "
            ).strip()
    
            proceso_area = input(
                "Proceso del área: "
            ).strip()
    
            observaciones = input(
                "Observaciones (opcional): "
            ).strip()
    
            try:
                actividad = {
                    "nombre": nombre,
                    "inicio": None,
                    "fin": None,
                    "duracion_min": int(duracion_min),
    
                    "frecuencia": frecuencia,
                    "volumen": int(volumen),
                    "autonomia": int(autonomia),
                    "proceso_area": proceso_area,
                    "observaciones": observaciones,
    
                    # Estos dos quedan pendientes de IA o edición manual
                    # "unidad_medida": "N/A",
                    # "phva": "SIN CLASIFICAR"
                }
    
                actividades.append(actividad)
                print("✅ Actividad agregada correctamente\n")
    
            except ValueError:
                print("⚠️ Error en los valores numéricos. Intente nuevamente.\n")
    
        return actividades

    def fusionar_inputs_usuario_nodiarias(self, actividades_base, actividades_enriquecidas):
        """
        Une actividades detectadas + enriquecidas por IA (no diarias)
        """
        actividades_finales_nodiarias = []
        
        for base, ia in zip(
            actividades_base,
            actividades_enriquecidas
        ):
            actividades_finales_nodiarias.append({
                "nombre": base["nombre"],
                "inicio": base["inicio"],
                "fin": base["fin"],
                "duracion_min": base["duracion_min"],
        
                "unidad_medida": ia["unidad_medida"],
                "phva": ia["phva"],
        
                "frecuencia": base["frecuencia"],
                "volumen": base["volumen"],
                "proceso_area": base["proceso_area"],
                "autonomia": base["autonomia"],
                "observaciones": base.get("observaciones", "")
            })
        
        return actividades_finales_nodiarias


#### Parte del Usuario

In [109]:
def main():
    print("\n📋 ANALIZADOR DE ACTIVIDADES DESDE TEAMS")

    upload = widgets.FileUpload(
        accept=".vtt",
        multiple=False
    )

    display(upload)
    return upload

In [110]:
upload_widget = main()


📋 ANALIZADOR DE ACTIVIDADES DESDE TEAMS


FileUpload(value=(), accept='.vtt', description='Upload')

In [111]:
procesador = ProcesadorTranscripcionTeams()

In [112]:
contexto_widgets = procesador.capturar_contexto_usuario()

Text(value='', description='Cargo:', layout=Layout(width='400px'), placeholder='Ej: Analista de Datos', style=…

Text(value='', description='Vicepresidencia:', layout=Layout(width='400px'), placeholder='Ej: Tecnología', sty…

In [113]:
contexto = procesador.obtener_contexto_valores(contexto_widgets)

In [114]:
contexto

{'cargo': 'Gerente ', 'vicepresidencia': 'Vp de Talento y Administrativa'}

In [115]:
contenido = procesador.obtener_contenido_vtt(upload_widget)
actividades_base = procesador.procesar_archivo(contenido)

In [116]:
actividades_base

[{'actividad': 'Voy a iniciar la actividad de Revision de plan de trabajo individual y envio correo.',
  'inicio': '00:00:00',
  'fin': '00:35:00',
  'duracion_min': 35},
 {'actividad': 'Voy a iniciar la actividad de Revision de plan de trabajo individual y envio correo.',
  'inicio': '01:00:00',
  'fin': '01:35:00',
  'duracion_min': 35},
 {'actividad': 'Voy a iniciar la actividad de Revision de plan de trabajo individual y envio correo.',
  'inicio': '04:00:00',
  'fin': '04:35:00',
  'duracion_min': 35}]

In [117]:
# inputs_usuario_widgets = []

# for actividad in actividades_base:
#     print(f"\n📝 Actividad: {actividad['actividad']}")
    
#     widgets_act = procesador.capturar_inputs_usuario()
#     inputs_usuario_widgets.append(widgets_act)

# inputs_por_actividad = procesador.capturar_inputs_usuario_actividades_unicas(
#     actividades_base
# )

inputs_usuario_widgets = []

actividades_unicas = procesador.obtener_actividades_unicas(actividades_base)

for actividad in actividades_unicas:
    print(f"\n📝 Actividad: {actividad['actividad']}")
    
    widgets_act = procesador.capturar_inputs_usuario()
    inputs_usuario_widgets.append(widgets_act)
    #inputs_usuario_widgets[actividad["actividad"].strip().lower()] = widgets_act


📝 Actividad: Voy a iniciar la actividad de Revision de plan de trabajo individual y envio correo.


Dropdown(description='Frecuencia:', options=('Diario', 'Mensual', 'Semanal', 'Quincenal', 'Bimensual', 'Trimes…

IntText(value=1, description='Volumen:', style=DescriptionStyle(description_width='120px'))

IntSlider(value=80, description='Autonomía (%):', style=SliderStyle(description_width='120px'))

Textarea(value='', description='Proceso del área:', layout=Layout(height='80px', width='500px'), style=TextSty…

Textarea(value='', description='Observaciones:', layout=Layout(height='80px', width='500px'), style=TextStyle(…

In [118]:
inputs_usuario_widgets

[{'frecuencia': Dropdown(description='Frecuencia:', options=('Diario', 'Mensual', 'Semanal', 'Quincenal', 'Bimensual', 'Trimestral', 'Semestral', 'Anual'), style=DescriptionStyle(description_width='120px'), value='Diario'),
  'volumen': IntText(value=3, description='Volumen:', style=DescriptionStyle(description_width='120px')),
  'proceso_area': Textarea(value='Desarrollo institucional de nuevas formas de trabajo y agilidad', description='Proceso del área:', layout=Layout(height='80px', width='500px'), style=TextStyle(description_width='120px')),
  'autonomia': IntSlider(value=100, description='Autonomía (%):', style=SliderStyle(description_width='120px')),
  'observaciones': Textarea(value='Revision de pendientes y nuevas solicitudes', description='Observaciones:', layout=Layout(height='80px', width='500px'), style=TextStyle(description_width='120px'))}]

In [64]:
actividades_no_diarias = procesador.capturar_actividades_no_diarias()


📌 REGISTRO DE ACTIVIDADES NO DIARIAS
Escriba 'salir' cuando no desee agregar más actividades.



Nombre de la actividad:  Liderar  el diseño y actualización del roadmap de  Delivery en Negocio
Frecuencia (Semanal / Mensual / Trimestral / etc.):  Semanal
Duración estimada por ejecución (en minutos):  60
Volumen por periodo (ej: 1, 2, 5):  1
Autonomía (%):  100
Proceso del área:  Consolidación del equipo de Cultura y Transformación
Observaciones (opcional):  Sesion de alineación de prioridades con el equipo de leads de agilidad 


✅ Actividad agregada correctamente



Nombre de la actividad:  Liderar  el diseño y actualización del roadmap de transformación organizacional
Frecuencia (Semanal / Mensual / Trimestral / etc.):  Semanal
Duración estimada por ejecución (en minutos):  120
Volumen por periodo (ej: 1, 2, 5):  1
Autonomía (%):  65
Proceso del área:  Consolidación del equipo de Cultura y Transformación
Observaciones (opcional):  Alineación Lideres Dirección Cultura y Transformación Organizacional,Alineacion y presentacion de avances a la estrategia de cultura y tranformación


✅ Actividad agregada correctamente



...

Nombre de la actividad:  Formacion continua del rol
Frecuencia (Semanal / Mensual / Trimestral / etc.):  Semanal
Duración estimada por ejecución (en minutos):  120
Volumen por periodo (ej: 1, 2, 5):  1
Autonomía (%):  100
Proceso del área:  Desarrollo institucional de nuevas formas de trabajo y agilidad
Observaciones (opcional):  Benchmark de buenas practicas , consulta de portales como gartner y mckinsey, estudios en temas de vanguardia en transformación 


✅ Actividad agregada correctamente



Nombre de la actividad:  Definir estrategias de gestión del cambio para iniciativas clave
Frecuencia (Semanal / Mensual / Trimestral / etc.):  Semanal
Duración estimada por ejecución (en minutos):  120
Volumen por periodo (ej: 1, 2, 5):  1
Autonomía (%):  50
Proceso del área:  Desarrollo institucional de nuevas formas de trabajo y agilidad
Observaciones (opcional):  Definir estrategias de gestión del cambio para iniciativas clave


✅ Actividad agregada correctamente



Nombre de la actividad:  salir


In [119]:
actividades_no_diarias

[{'nombre': 'Liderar  el diseño y actualización del roadmap de  Delivery en Negocio',
  'inicio': None,
  'fin': None,
  'duracion_min': 60,
  'frecuencia': 'Semanal',
  'volumen': 1,
  'autonomia': 100,
  'proceso_area': 'Consolidación del equipo de Cultura y Transformación',
  'observaciones': 'Sesion de alineación de prioridades con el equipo de leads de agilidad'},
 {'nombre': 'Liderar  el diseño y actualización del roadmap de transformación organizacional',
  'inicio': None,
  'fin': None,
  'duracion_min': 120,
  'frecuencia': 'Semanal',
  'volumen': 1,
  'autonomia': 65,
  'proceso_area': 'Consolidación del equipo de Cultura y Transformación',
  'observaciones': 'Alineación Lideres Dirección Cultura y Transformación Organizacional,Alineacion y presentacion de avances a la estrategia de cultura y tranformación'},
 {'nombre': 'Revisión del roadmap de delivery con el equipo de agilidad (Empresas, retail,riesgo)',
  'inicio': None,
  'fin': None,
  'duracion_min': 60,
  'frecuencia'

In [120]:
inputs_usuario = []

for widgets_act in inputs_usuario_widgets:
    valores = procesador.obtener_inputs_usuario_valores(widgets_act)
    inputs_usuario.append(valores)

In [121]:
inputs_usuario

[{'frecuencia': 'Diario',
  'volumen': 3,
  'autonomia': 100,
  'proceso_area': 'Desarrollo institucional de nuevas formas de trabajo y agilidad',
  'observaciones': 'Revision de pendientes y nuevas solicitudes'}]

In [122]:
actividades_ia = procesador.enriquecer_actividades(actividades_base, cargo=contexto["cargo"], vicepresidencia=contexto["vicepresidencia"])
actividades_ia_nodiarias = procesador.enriquecer_actividades(actividades_no_diarias, cargo=contexto["cargo"], vicepresidencia=contexto["vicepresidencia"])

In [123]:
actividades_finales = procesador.fusionar_inputs_usuario(
    actividades_base,
    actividades_ia,
    inputs_usuario
)

In [124]:
actividades_finales_nodiarias = procesador.fusionar_inputs_usuario_nodiarias(
    actividades_no_diarias,
    actividades_ia_nodiarias
)

In [125]:
actividades_finales_final = actividades_finales + actividades_finales_nodiarias

In [126]:
resumen = procesador.resumen_para_confirmacion(
    actividades_finales_final,
    contexto
)

In [127]:
if procesador.confirmar_informacion(resumen):
    metricas = procesador.calcular_metricas(actividades_finales_final)
    analisis = procesador.analisis_analista_ia(
        actividades_finales_final,
        metricas,
        contexto
    )
else:
    print("✏️ El usuario solicitó ajustes.")
    procesador.editar_actividades(actividades_finales_final)

📋 RESUMEN PARA CONFIRMACIÓN

Resumen de actividades del colaborador (Gerente, Vicepresidencia de Talento y Administrativa):

1. Revisión de plan de trabajo individual y envío de correo  
- Frecuencia: Diario  
- Duración: 35 minutos  
- Volumen: 3 informes  
- Proceso: Desarrollo institucional de nuevas formas de trabajo y agilidad  
- Tipo de actividad: Verificar  
- Autonomía: 100%  
- Observaciones: Revisión de pendientes y nuevas solicitudes  

2. Liderar diseño y actualización del roadmap de Delivery en Negocio  
- Frecuencia: Semanal  
- Duración: 60 minutos  
- Volumen: 1 reunión  
- Proceso: Consolidación del equipo de Cultura y Transformación  
- Tipo de actividad: Planear  
- Autonomía: 100%  
- Observaciones: Sesión de alineación de prioridades con el equipo de leads de agilidad  

3. Liderar diseño y actualización del roadmap de transformación organizacional  
- Frecuencia: Semanal  
- Duración: 120 minutos  
- Volumen: 1 reunión  
- Proceso: Consolidación del equipo de Cul


¿La información es correcta? (si / no):  no


✏️ El usuario solicitó ajustes.

✏️ ACTIVIDADES DISPONIBLES:
1. Revisión de plan de trabajo individual y envío de correo
2. Liderar  el diseño y actualización del roadmap de  Delivery en Negocio
3. Liderar  el diseño y actualización del roadmap de transformación organizacional
4. Revisión del roadmap de delivery con el equipo de agilidad (Empresas, retail,riesgo)
5. Revisión del roadmap de delivery con el equipo de agilidad(Canales , Cash , Internacional)
6. Revisión del roadmap de delivery con el equipo de agilidad(Canales Fisicos , Medios de pago , Seguros)
7. Revisión del roadmap de delivery con el equipo de agilidad (Apis, core bancario, informó transaccional,Backofice)
8. Sesion de alineación de agilidad con aval (adl)
9. Agile GO Frente personas
10. Agile GO Frente empresas
11. Uno a Uno con lideres de agilidad
12. Sesión mensual de dirección estratégica de la transformación
13. Alineación de obejtivos organizacionales
14. Feria Bdb
15. steerco
16. Gestión de iniciativas estratég


Ingrese el número de la actividad a modificar (o 'salir' para terminar ajustes):  1



Campos editables:
- frecuencia (actual: Diario)
- volumen (actual: 3)
- duracion_min (actual: 35)
- autonomia (actual: 100)
- observaciones (actual: Revision de pendientes y nuevas solicitudes)
- unidad_medida (actual: informes)
- phva (actual: Verificar)


...


Ingrese el número de la actividad a modificar (o 'salir' para terminar ajustes):  18



Campos editables:
- frecuencia (actual: Semanal)
- volumen (actual: 1)
- duracion_min (actual: 120)
- autonomia (actual: 50)
- observaciones (actual: Definir estrategias de gestión del cambio para iniciativas clave)
- unidad_medida (actual: horas)
- phva (actual: Planear)



Campo a modificar:  salir


⚠️ Campo no editable

✏️ ACTIVIDADES DISPONIBLES:
1. Revisión de plan de trabajo individual y envío de correo
2. Liderar  el diseño y actualización del roadmap de  Delivery en Negocio
3. Liderar  el diseño y actualización del roadmap de transformación organizacional
4. Revisión del roadmap de delivery con el equipo de agilidad (Empresas, retail,riesgo)
5. Revisión del roadmap de delivery con el equipo de agilidad(Canales , Cash , Internacional)
6. Revisión del roadmap de delivery con el equipo de agilidad(Canales Fisicos , Medios de pago , Seguros)
7. Revisión del roadmap de delivery con el equipo de agilidad (Apis, core bancario, informó transaccional,Backofice)
8. Sesion de alineación de agilidad con aval (adl)
9. Agile GO Frente personas
10. Agile GO Frente empresas
11. Uno a Uno con lideres de agilidad
12. Sesión mensual de dirección estratégica de la transformación
13. Alineación de obejtivos organizacionales
14. Feria Bdb
15. steerco
16. Gestión de iniciativas estratégicas transv


Ingrese el número de la actividad a modificar (o 'salir' para terminar ajustes):  salir


In [128]:
actividades_finales_final

[{'nombre': 'Revisión de plan de trabajo individual y envío de correo',
  'inicio': '00:00:00',
  'fin': '00:35:00',
  'duracion_min': 35,
  'unidad_medida': 'Correo',
  'phva': 'Hacer',
  'frecuencia': 'Diario',
  'volumen': 3,
  'proceso_area': 'Desarrollo institucional de nuevas formas de trabajo y agilidad',
  'autonomia': 100,
  'observaciones': 'Revision de pendientes y nuevas solicitudes'},
 {'nombre': 'Liderar  el diseño y actualización del roadmap de  Delivery en Negocio',
  'inicio': None,
  'fin': None,
  'duracion_min': 60,
  'unidad_medida': 'Comite Primario',
  'phva': 'Planear',
  'frecuencia': 'Semanal',
  'volumen': 1,
  'proceso_area': 'Consolidación del equipo de Cultura y Transformación',
  'autonomia': 100,
  'observaciones': 'Sesion de alineación de prioridades con el equipo de leads de agilidad'},
 {'nombre': 'Liderar  el diseño y actualización del roadmap de transformación organizacional',
  'inicio': None,
  'fin': None,
  'duracion_min': 120,
  'unidad_medida'

#### Parte del Analista

In [129]:
resumen = procesador.resumen_para_confirmacion(
    actividades_finales_final,
    contexto
)

In [130]:
metricas = procesador.calcular_metricas(actividades_finales_final)

In [131]:
metricas

{'carga_trabajo_phva': {'Hacer': 0.167,
  'Planear': 0.2045,
  'Verificar': 0.0477,
  'Actuar': 0.1618},
 'cantidad_actividades_phva': {'Hacer': 1,
  'Planear': 9,
  'Verificar': 5,
  'Actuar': 3},
 'porcentaje_actividades_phva': {'Hacer': 0.2874,
  'Planear': 0.352,
  'Verificar': 0.0821,
  'Actuar': 0.2785},
 'carga_trabajo_frecuencia': {'Diario': 0.167,
  'Semanal': 0.3436,
  'Quincenal': 0.0477,
  'Mensual': 0.0091,
  'Trimestral': 0.0136},
 'cantidad_actividades_frecuencia': {'Diario': 1,
  'Semanal': 8,
  'Quincenal': 5,
  'Mensual': 1,
  'Trimestral': 3},
 'porcentaje_actividades_frecuencia': {'Diario': 0.2874,
  'Semanal': 0.5913,
  'Quincenal': 0.0821,
  'Mensual': 0.0156,
  'Trimestral': 0.0235},
 'carga_trabajo_proceso_area': {'Desarrollo institucional de nuevas formas de trabajo y agilidad': 0.5239,
  'Consolidación del equipo de Cultura y Transformación': 0.0573},
 'cantidad_actividades_proceso_area': {'Desarrollo institucional de nuevas formas de trabajo y agilidad': 16,


In [132]:
analisis = procesador.analisis_analista_ia(
        actividades_finales_final,
        metricas,
        contexto
    )

In [133]:
display(Markdown("## 📋 Actividades"))

df_act = pd.DataFrame(actividades_finales_final)
display(df_act)

display(Markdown("## 📊 Métricas por PHVA"))
df_phva = pd.DataFrame({
    "Carga de Trabajo": metricas["carga_trabajo_phva"],
    "# Act.": metricas["cantidad_actividades_phva"],
    "% Act.": metricas["porcentaje_actividades_phva"]
})
display(df_phva)

display(Markdown("## 📊 Métricas por FRECUENCIA"))
df_frecuencia = pd.DataFrame({
    "Carga de Trabajo": metricas["carga_trabajo_frecuencia"],
    "# Act.": metricas["cantidad_actividades_frecuencia"],
    "% Act.": metricas["porcentaje_actividades_frecuencia"]
})
display(df_frecuencia)

display(Markdown("## 📊 Métricas por PROCESO"))
df_proceso = pd.DataFrame({
    "Carga de Trabajo": metricas["carga_trabajo_proceso_area"],
    "# Act.": metricas["cantidad_actividades_proceso_area"],
    "% Act.": metricas["porcentaje_actividades_proceso_area"]
})
display(df_proceso)

display(Markdown("## 📊 TOTALES"))
df_totales = pd.DataFrame([{
    "Total Carga W sin TM": metricas["total_carga_w_sin_tm"],
    "Total Carga Trabajo Individual": metricas["total_carga_trabajo_individual"],
    "Minutos diarios empleados": metricas["minutos_diarios_empleados"],
    "Horas diarias requeridas": metricas["horas_diarias_requeridas"]
}])
display(df_totales)

display(Markdown("## 📊 DOTACIÓN"))
df_dotacion = pd.DataFrame([{
    "Jornada Laboral Diaria": metricas["jornada_laboral_diaria"],
    "Factor Tiempo Neto Productivo": metricas["factor_tiempo_neto_productivo"],
    "Horas Netas Efectivas por Persona": metricas["horas_netas_efectivas_por_persona"],
    "Número Personas Requeridas": metricas["numero_personas_requeridas"]
}])
display(df_dotacion)

display(Markdown("## 📊 AJUSTE FINAL"))
df_ajuste = pd.DataFrame([{
    "Tiempo Muerto": metricas["tiempo_muerto"],
    "Horas Diarias Requeridas Final": metricas["horas_diarias_requeridas_final"]
}])
display(df_ajuste)

display(Markdown("## 🧠 Análisis del Analista IA"))
display(Markdown(analisis))

## 📋 Actividades

,nombre,inicio,fin,duracion_min,unidad_medida,phva,frecuencia,volumen,proceso_area,autonomia,observaciones,metricas
0,Revisión de plan de trabajo individual y envío...,00:00:00,00:35:00,35,Correo,Hacer,Diario,3,Desarrollo institucional de nuevas formas de t...,100,Revision de pendientes y nuevas solicitudes,"{'minutos_diarios': 105, 'minutos_mes': 2205, ..."
1,Liderar el diseño y actualización del roadmap...,None,None,60,Comite Primario,Planear,Semanal,1,Consolidación del equipo de Cultura y Transfor...,100,Sesion de alineación de prioridades con el equ...,"{'minutos_diarios': 12.0, 'minutos_mes': 252.0..."
2,Liderar el diseño y actualización del roadmap...,None,None,120,Comite Primario,Planear,Semanal,1,Consolidación del equipo de Cultura y Transfor...,65,Alineación Lideres Dirección Cultura y Transfo...,"{'minutos_diarios': 24.0, 'minutos_mes': 504.0..."
3,Revisión del roadmap de delivery con el equipo...,None,None,60,reuniones,Verificar,Quincenal,1,Desarrollo institucional de nuevas formas de t...,30,Revision de avances en plan de trabajo de agil...,"{'minutos_diarios': 6.0, 'minutos_mes': 126.0,..."
4,Revisión del roadmap de delivery con el equipo...,None,None,60,reuniones,Verificar,Quincenal,1,Desarrollo institucional de nuevas formas de t...,30,Revision de avances en plan de trabajo de agil...,"{'minutos_diarios': 6.0, 'minutos_mes': 126.0,..."
5,Revisión del roadmap de delivery con el equipo...,None,None,60,reuniones,Verificar,Quincenal,1,Desarrollo institucional de nuevas formas de t...,30,Revision de avances en plan de trabajo de agil...,"{'minutos_diarios': 6.0, 'minutos_mes': 126.0,..."
6,Revisión del roadmap de delivery con el equipo...,None,None,60,reuniones,Verificar,Quincenal,1,Desarrollo institucional de nuevas formas de t...,30,Revision de avances en plan de trabajo de agil...,"{'minutos_diarios': 6.0, 'minutos_mes': 126.0,..."
7,Sesion de alineación de agilidad con aval (adl),None,None,60,reuniones,Verificar,Quincenal,1,Desarrollo institucional de nuevas formas de t...,20,Alineación de formas de trabajo con areas de a...,"{'minutos_diarios': 6.0, 'minutos_mes': 126.0,..."
8,Agile GO Frente personas,None,None,90,Reunión,Planear,Semanal,1,Desarrollo institucional de nuevas formas de t...,10,Seguimiento a la ejecución de la estrategia co...,"{'minutos_diarios': 18.0, 'minutos_mes': 378.0..."
9,Agile GO Frente empresas,None,None,90,Reuniones,Planear,Semanal,1,Desarrollo institucional de nuevas formas de t...,10,Seguimiento a la ejecución de la estrategia c...,"{'minutos_diarios': 18.0, 'minutos_mes': 378.0..."


## 📊 Métricas por PHVA

,Carga de Trabajo,# Act.,% Act.
Hacer,0.1670,1,0.2874
Planear,0.2045,9,0.3520
Verificar,0.0477,5,0.0821
Actuar,0.1618,3,0.2785


## 📊 Métricas por FRECUENCIA

...

,Jornada Laboral Diaria,Factor Tiempo Neto Productivo,Horas Netas Efectivas por Persona,Número Personas Requeridas
0,8.5,0.8676,7.37,0.826


## 📊 AJUSTE FINAL

,Tiempo Muerto,Horas Diarias Requeridas Final
0,0.1324,6.89


## 🧠 Análisis del Analista IA

1. **Nivel de carga laboral (con justificación)**

La carga laboral diaria final calculada para el cargo de Gerente en la Vicepresidencia de Talento y Administrativa es de **6.89 horas diarias** (horas_diarias_requeridas_final), dentro de una jornada laboral de **8.5 horas**. Esto representa aproximadamente un **81%** de la jornada laboral dedicada a actividades operativas y estratégicas, dejando un margen de tiempo para imprevistos, pausas y actividades no registradas.

La dotación requerida es de **0.826 personas**, lo que indica que la carga de trabajo es asumible por una sola persona sin necesidad de apoyo adicional, pero con un nivel de ocupación alto.

Por lo tanto, clasifico la carga laboral como **ADECUADA**, aunque cercana a un nivel alto, dado que la persona debe dedicar más de 6.8 horas diarias a actividades planificadas, lo que implica un ritmo intenso pero manejable para un cargo gerencial con alta autonomía.

---

2. **Validación de coherencia de las métricas**

- La carga total sin tiempos muertos es **0.5811** (58.11% de la jornada), y la carga individual ajustada es **0.4193** (41.93%), lo que refleja un ajuste razonable considerando tiempos muertos y actividades no productivas.

- La autonomía promedio declarada es alta, con valores que oscilan entre 10% y 100%, y un promedio ponderado que se estima alto (no dado explícitamente, pero muchas actividades tienen autonomía 100%). Esto es coherente con un cargo gerencial que tiene capacidad de decisión y control sobre sus actividades.

- No se identifican valores atípicos en duración o frecuencia. La actividad con mayor carga es la "Revisión de plan de trabajo individual y envío de correo" con 105 minutos diarios, lo cual es lógico para un gerente que debe revisar pendientes y coordinar.

- La concentración de actividades en el proceso "Desarrollo institucional de nuevas formas de trabajo y agilidad" (90.14% de las actividades y 52.39% de la carga) es coherente con el rol estratégico y de transformación del cargo.

- La frecuencia semanal domina con 59.13% de las actividades y 34.36% de la carga, lo que es típico en roles gerenciales que manejan reuniones y seguimiento constante.

En resumen, las métricas son consistentes con el perfil y responsabilidades del cargo.

---

3. **Interpretación del balance PHVA**

- La distribución de carga por PHVA es:  
  - Hacer: 16.7%  
  - Planear: 20.45%  
  - Verificar: 4.77%  
  - Actuar: 16.18%

- En cuanto a cantidad de actividades:  
  - Hacer: 28.74%  
  - Planear: 35.2%  
  - Verificar: 8.21%  
  - Actuar: 27.85%

El rol tiene un enfoque equilibrado entre Planear (20.45%) y Hacer (16.7%), con una presencia significativa de Actuar (16.18%) y menor en Verificar (4.77%).

Esto indica que el gerente está más orientado a la planificación estratégica y ejecución directa, con una adecuada participación en la mejora continua (Actuar), pero relativamente menos en la fase de Verificación o control.

No se observa sobregestión (exceso de Planear sin ejecución) ni baja ejecución, pero sí una posible oportunidad para fortalecer la fase de Verificar, que es la más baja, para asegurar el seguimiento y control efectivo de las iniciativas.

---

4. **Riesgos operativos identificados**

- **Riesgo de sobrecarga:** La carga diaria de casi 7 horas en actividades planificadas puede generar fatiga o limitación para atender imprevistos, especialmente considerando la alta frecuencia semanal de actividades (59.13%).

- **Dependencia:** La alta autonomía (varias actividades con 100%) reduce riesgos de dependencia, pero actividades con autonomía baja (10%-30%) en reuniones clave pueden generar cuellos de botella si el gerente no puede asistir o delegar.

- **Reprocesos:** La baja carga en Verificar (4.77%) puede implicar riesgo de reprocesos o falta de control riguroso sobre la ejecución de planes.

- **Frecuencia y volumen:** Varias actividades quincenales y semanales con duración considerable (60-120 minutos) pueden acumularse en días específicos, generando picos de carga.

- **Baja autonomía:** Algunas actividades de alta duración y frecuencia semanal tienen autonomía baja (10%-30%), lo que puede generar dependencia de otros actores y retrasos.

---

5. **Oportunidades de automatización o mejora**

- La actividad diaria de "Revisión de plan de trabajo individual y envío de correo" (105 minutos diarios, autonomía 100%) es repetitiva y de alta frecuencia, por lo que podría beneficiarse de herramientas que optimicen la gestión de pendientes y comunicación.

- Actividades con alta carga y baja autonomía, como las reuniones de alineación y seguimiento (autonomía 10%-30%), podrían ser optimizadas en agenda o delegadas parcialmente para liberar tiempo del gerente.

- La baja carga en Verificar sugiere oportunidad para implementar mecanismos automáticos de seguimiento y reporte que reduzcan la necesidad de reuniones presenciales o manuales.

- La formación continua (120 minutos semanales, autonomía 100%) es importante, pero podría complementarse con formatos más flexibles o integrados para optimizar tiempo.

---

6. **Recomendaciones finales**

- Mantener la carga laboral en niveles adecuados, vigilando que la alta ocupación diaria no afecte la capacidad de respuesta ante imprevistos o actividades no planificadas.

- Fortalecer la fase de Verificar para mejorar el control y seguimiento de iniciativas, evitando reprocesos y asegurando la calidad en la ejecución.

- Evaluar la redistribución o delegación de actividades con baja autonomía y alta duración para reducir riesgos de cuellos de botella y dependencia.

- Promover la optimización de actividades repetitivas y de alta frecuencia, especialmente la gestión de correos y revisión de pendientes, mediante lineamientos para uso eficiente de herramientas digitales.

- Fomentar la flexibilidad en la formación continua para maximizar el aprendizaje sin afectar la disponibilidad operativa.

- Finalmente, mantener un monitoreo periódico de la carga laboral y autonomía para ajustar la gestión del cargo conforme evolucionen las responsabilidades y el contexto organizacional.

In [134]:
dfs = {
    "Actividades": df_act,
    "PHVA": df_phva,
    "Dotacion": df_dotacion,
    "Ajuste": df_ajuste,
    "Frecuencia": df_frecuencia,
    "Totales": df_totales,
    "Proceso_Area": df_proceso
}

In [135]:
ruta_excel = f"analisis_carga_laboral_{contexto['cargo']}.xlsx"

with pd.ExcelWriter(ruta_excel, engine="openpyxl") as writer:
    df_act.to_excel(writer, sheet_name="Actividades", index=True)
    df_phva.to_excel(writer, sheet_name="Métricas PHVA", index=True)
    df_frecuencia.to_excel(writer, sheet_name="Frecuencias", index=True)
    df_proceso.to_excel(writer, sheet_name="Procesos", index=True)
    df_dotacion.to_excel(writer, sheet_name="Dotacion", index=False)
    df_ajuste.to_excel(writer, sheet_name="Ajuste", index=False)
    df_totales.to_excel(writer, sheet_name="Totales", index=False)

In [136]:
wb = load_workbook(ruta_excel)

for sheet in wb.sheetnames:
    ws = wb[sheet]
    for col in ws.columns:
        ws.column_dimensions[col[0].column_letter].width = 25

wb.save(ruta_excel)

In [137]:
display(HTML(f"""
<a href="{ruta_excel}" download>
📥 Descargar archivo Excel de análisis de carga laboral
</a>
"""))

In [ ]:
# question:str = '¿Qué es un LLM?'

# prompt_template:str = """
#     Responde la pregunta del usuario en rimas
#     __
#     Pregunta del usuario: {question}
# """

# prompt:str = prompt_template.format(
#     question=question
# )

# reply = llm.invoke(prompt)
# response = reply.content
# print(response)

# Caso de Uso

## Capturar información del usuario

In [ ]:
tiempo=''
area=''
vp=''

## Formatear el prompt

In [ ]:
prompt_template:str="""
    Eres un ....
    Tu objetivo es ...

    tiempo: {tiempo}
    vp: {vp}
"""

prompt:str = prompt_template.format(
    tiempo=tiempo,
    vp=vp
)

## Analizar la información

In [ ]:
reply = llm.invoke(prompt)
response = reply.content
print(response)